In [24]:
import pandas as pd
import polars as pl
import numpy as np

In [2]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed

)

In [3]:
from datasets import load_dataset

In [4]:
import torch

In [13]:
from sklearn.metrics import accuracy_score

##BERT での穴埋め問題

In [5]:
device = 0 if torch.cuda.is_available() else -1

mask_pipeline = pipeline(
    "fill-mask",
    model="cl-tohoku/bert-base-japanese-v3",
    device=device,
)

display(mask_pipeline("Pythonは[MASK]言語です。")[0])

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'score': 0.7647461891174316,
 'token': 20656,
 'token_str': 'プログラミング',
 'sequence': 'Python は プログラミング 言語 です 。'}

In [6]:
display(mask_pipeline("織田信長は[MASK]です。")[0])

{'score': 0.14428521692752838,
 'token': 25648,
 'token_str': 'フィクション',
 'sequence': '織田 信長 は フィクション です 。'}

##GPTを動かすでー

In [7]:
set_seed(13)
device = 0 if torch.cuda.is_available() else -1

gen_pipeline = pipeline(
    "text-generation",
    model="abeja/gpt-neox-japanese-2.7b",
    tokenizer="abeja/gpt-neox-japanese-2.7b",
    device=device,
)

Loading weights:   0%|          | 0/261 [00:00<?, ?it/s]

In [8]:
prompt = "Pythonの特徴は"
result = gen_pipeline(prompt)
print(result[0]["generated_text"])

Pythonの特徴は、プログラミング言語がPythonであるだけでなく、Pythonで開発されたフレームワークが多く存在する点です。


In [9]:
prompt = "JavaScriptの特徴は"
result = gen_pipeline(prompt)
print(result[0]["generated_text"])

JavaScriptの特徴は、インタラクティブなコードを書くことができる点にあります。


In [10]:
from transformers import AutoTokenizer

model_name = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "カフェでコーヒーを飲みました。"
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("tokens:", tokens)
print("ids:", ids)

tokens: ['カフェ', 'で', 'コーヒー', 'を', '飲み', 'まし', 'た', '。']
ids: [17499, 457, 18423, 500, 18148, 14347, 449, 385]


##感情分析

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REPO_ID = "tetsuro731/wrime-sentiment"
train_dataset = load_dataset(REPO_ID, split="train")
valid_dataset = load_dataset(REPO_ID, split="validation")

print(f"train: {len(train_dataset)}件")
print(f"validation: {len(valid_dataset)}件")

for i in range(5):
    display(train_dataset[i])

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.71MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  177kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  176kB            

... (6 outputs left out)

{'sentence': '今日の月も白くて明るい。昨日より雲が少なくてキレイな〜 と立ち止まる帰り道。チャリなし生活も悪くない。',
 'label': 0,
 'user_id': 1,
 'datetime': '2012/8/2 23:09'}

{'sentence': '眠い、眠れない。', 'label': 1, 'user_id': 1, 'datetime': '2012/8/8 1:36'}

{'sentence': '表情筋が衰えてきてる。まずいな…',
 'label': 1,
 'user_id': 1,
 'datetime': '2012/8/11 1:20'}

{'sentence': 'やばい。おもろいな〜思ってみてみた「シャレードがいっぱい」。よすぎるやん。',
 'label': 0,
 'user_id': 1,
 'datetime': '2012/8/16 23:50'}

In [15]:
#model_make
model_name = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    label2id={"positive": 0, "negative": 1},
    id2label={0: "positive", 1: "negative"},
    ).to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [16]:
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [17]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)


Parameter 'function'=<function tokenize_function at 0x00000239F5A491B0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/20149 [00:00<?, ? examples/s]

Map:   0%|          | 0/1608 [00:00<?, ? examples/s]

In [18]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

In [21]:
training_args = TrainingArguments(
    output_dir="./wrime_sentiment_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=42,
    report_to=[]
)


In [22]:
#training and evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
eval_result = trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.138098,0.223574,0.934080
2,0.061067,0.280961,0.937189
3,0.025861,0.332387,0.935323


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\rally\python\uv-workspace\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\rally\python\uv-workspace\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\rally\python\uv-workspace\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.025861,0.280961,3,0.937189


ValueError: Unknown format code 'f' for object of type 'str'